# Tree Round-trip Test

Load `treetest.json` (a v1-format tree exported from the UI), upgrade it to v2, convert to a
`PrioritizedTreeModule` for execution, run test data through it, then convert it back to a v2 UI
tree to verify the round-trip.

**Tree layout (from the JSON)**
```
Subtree 1 (priority 1) — root: StringMatchNode(feature_0, ["10","20"])
  ├─ "10"  → RangeNode(feature_0, thresholds=[0,10,30,40])
  │           └─ >40  → Leaf (Action2 / number=10)
  │           └─ ≤40  → Leaf (-1, no match)
  ├─ "20"  → Leaf (Action1 / number=99)
  └─ other → Leaf (Action3 / number=12)          ← always matches

Subtree 2 (priority 2) — root: NumericalNode(feature_1 ≤ 5)
  ├─ ≤ 5  → Leaf (Action1 / number=10)
  └─ > 5  → CategoricalNode(feature_1 in [1,2,3,4])
              ├─ in  → RangeNode(feature_2, thresholds=[1,2,3])
              │         └─ >3 → Leaf (Action3 / number=5)
              └─ out → Leaf (-1, no match)
```

In [1]:
import json, sys, pathlib
import polars as pl

# Make sure the project root is on the path
ROOT = pathlib.Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# v1 / v2 UI tree models
from ped.modules.tree.ui.v1.tree import Tree as V1Tree
from ped.modules.tree.ui.v2.tree import Tree as V2Tree

# Execution layer
from ped.modules.tree.module import PrioritizedTreeModule
from ped.modules.tree.impl import execute_tree_on_frame, execute_prioritized_trees
from ped.serializable.function import DefinedFunction

print("Imports OK")

/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


Imports OK


/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "DataFrame" shadows an attribute in parent "BaseModel"
  warnings.warn(
/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "WithTreeOutput" shadows an attribute in parent "BaseModel"
  warnings.warn(


## 1 · Load the exported JSON and parse as a v1 tree

In [2]:
raw = json.loads((ROOT / "testing" / "treetest.json").read_text())
v1_tree = V1Tree.model_validate(raw)

print(f"Features : {v1_tree.features}")
print(f"Nodes    : {len(v1_tree.nodes)}")
print(f"Edges    : {len(v1_tree.edges)}")
print(f"Subtrees : {[s.rootNodeId for s in v1_tree.subtrees]}")
print(f"Output fmt version: {v1_tree.node_output_format_version}")

Features : ['feature_0', 'feature_1', 'feature_2']
Nodes    : 10
Edges    : 8
Subtrees : ['node_1755083290333', 'node_1773047407334']
Output fmt version: 1


## 2 · Upgrade to v2

In [3]:
v2_tree: V2Tree = v1_tree.upgrade()

print(f"Subtree root IDs : {v2_tree.subtrees}")
print(f"Output rows      : {len(v2_tree.output)}")
print(f"Default          : {v2_tree.default}")
print()
print("Collected output table:")
for i, row in enumerate(v2_tree.output):
    print(f"  [{i}]", {k: v for k, v in row.items() if k != "Action"}, "| Action id:", row.get("Action", {}).get("id"))

Subtree root IDs : ['node_1755083290333', 'node_1773047407334']
Output rows      : 5
Default          : {'Action': {'id': 2, 'Type': 'NCALimitPercentage', 'Amount': 0.1, 'Recipient': 'BANKS'}, 'number': 10, 'Outcome': 'Action1'}

Collected output table:
  [0] {'number': 10, 'Outcome': 'Action1'} | Action id: 2
  [1] {'number': 99, 'Outcome': 'Action1'} | Action id: 1
  [2] {'number': 12, 'Outcome': 'Action3'} | Action id: 1
  [3] {'number': 10, 'Outcome': 'Action1'} | Action id: 3
  [4] {'number': 5, 'Outcome': 'Action3'} | Action id: 1


In [4]:
print(v2_tree.to_tree_module().model_dump_json(indent=2))

{
  "type": "prioritized_tree",
  "name": "test",
  "input_mapping": {},
  "input_name": "input",
  "output_name": "result",
  "output": [
    {
      "Action": {
        "id": 2,
        "Type": "NCALimitPercentage",
        "Amount": 0.1,
        "Recipient": "BANKS"
      },
      "number": 10,
      "Outcome": "Action1"
    },
    {
      "Action": {
        "id": 1,
        "Type": "NCALimitAmount",
        "Amount": 100,
        "Recipient": "BANKS"
      },
      "number": 99,
      "Outcome": "Action1"
    },
    {
      "Action": {
        "id": 1,
        "Type": "NCALimitAmount",
        "Amount": 100,
        "Recipient": "BANKS"
      },
      "number": 12,
      "Outcome": "Action3"
    },
    {
      "Action": {
        "id": 3,
        "Type": "VMaxAmount",
        "Amount": 200,
        "Recipient": "VMAX"
      },
      "number": 10,
      "Outcome": "Action1"
    },
    {
      "Action": {
        "id": 1,
        "Type": "NCALimitAmount",
        "Amount": 100,
    

## 3 · Convert to execution module & inspect graph

In [5]:
tree_module = v2_tree.to_tree_module()

print(f"Module name : {tree_module.name!r}")
print(f"Output node : {tree_module.name}.{tree_module.output_name}")
print(f"Roots       : {len(tree_module.roots)}")

tree_module.as_constructed_graph_modules().build_graph(output_nodes=[]).hamilton_driver

-------------------------------------------------------------------
Oh no an error! Need help with Hamilton?
Join our slack and ask for help! https://join.slack.com/t/hamilton-opensource/shared_invite/zt-2niepkra8-DGKGf_tTYhXuJWBTXtIs4g
-------------------------------------------------------------------



Module name : 'test'
Output node : test.result
Roots       : 2


AttributeError: 'NoneType' object has no attribute '__name__'

## 4 · Build test data

The tree branches on three features:

| column | type | notes |
|---|---|---|
| `feature_0` | `Utf8` | StringMatch — `"10"` / `"20"` / other |
| `feature_1` | `Float64` | Numerical threshold ≤ 5 |
| `feature_2` | `Float64` | Range buckets |

In [ ]:
import polars as pl

data = pl.LazyFrame(
    {
        # Subtree-1 branches on feature_0 (StringMatch: "10" / "20" / other)
        # Subtree-2 branches on feature_1 (Numerical ≤ 5) then feature_2 ranges
        "feature_0": ["10",  "20",  "99",  "10",  "20",  "99"],
        "feature_1": [ 3.0,   7.0,   2.0,   6.0,   4.0,   8.0],
        "feature_2": [ 1.5,   0.5,   2.5,   3.5,   1.0,   4.0],
    },
    schema={
        "feature_0": pl.Utf8,
        "feature_1": pl.Float64,
        "feature_2": pl.Float64,
    },
)

data.collect()

## 5 · Execute & inspect result

In [ ]:
result = tree_module.execute(
    inputs={
        "feature_0": pl.col("feature_0"),
        "feature_1": pl.col("feature_1"),
        "feature_2": pl.col("feature_2"),
    }
)

output_key = f"{tree_module.name}.{tree_module.output_name}"
print(f"Output key: {output_key!r}")

data.with_columns(result[output_key].alias("tree_output")).collect()

In [ ]:
# Unnest the struct output into individual columns for a cleaner view
(
    data
    .with_columns(result[output_key].alias("tree_output"))
    .unnest("tree_output")
    .collect()
)